# Predicting Premier League Goals from Shot Location
### Exploratory data analysis — 2015–2016 English Premier League
all data and questions I got and Had chat organize

**Research question:** How does the location of a shot affect the likelihood that it becomes a goal? This is an exploratory step toward a future expected goals (xG) model

A shot is one observation. Its outcome is encoded as `goal = 1` when the shot scored and `goal = 0` otherwise. The xG model proposed for later work would estimate the probability of `goal = 1` for a new shot.

## 1. Introduction and background

Soccer teams take shots from many different locations, but shots do not all have the same chance of scoring. An expected goals (xG) model estimates the probability that a shot results in a goal. I am interested in whether location alone helps explain goal-scoring chances in the Premier League. I chose this question because it combines soccer with statistics and machine learning, and it could help compare the quality of chances teams create rather than only counting shots.

In this project I want to understand the shot-location data, visualize where shots and goals happen, and identify initial patterns. Later, I would like to use logistic regression to predict goal probabilities from shot locations

## 2. Data description and source

**Source:** [StatsBomb Open Data](https://github.com/statsbomb/open-data), competition ID `2` (Premier League), season ID `27` (2015/2016). Match metadata is stored in `data/matches/2/27.json`; each match's event records are stored in `data/events/{match_id}.json`. The notebook (or accompanying `download_shots.py`) collects events whose `type.name` is `Shot`, reads their `location`, and marks `shot.outcome.name == "Goal"` as 1. All other shot outcomes become 0. A goal is not the same as a shot on target; misses, saves, and blocks are all 0.

**Unit of analysis:** One recorded shot event. The downloaded file `premier_league_2015_2016_shots.csv` includes the x and y location, binary goal label, match ID, team, original outcome and shot type. The primary analysis uses x, y and goal; other columns are retained for traceability and data checks, not prediction. StatsBomb uses a **120 × 80 coordinate grid**; attacking goal center is at approximately `(120, 40)`. Coordinates are not meters. Distance calculated from these coordinates is in *coordinate units*, not physical meters.

**Ethics and limitations:** These are historical match-event observations, not personal sensitive data. Still, data providers deserve credit and license compliance. StatsBomb asks for attribution when publishing analyses based on its open data. Event locations may be imperfect, and location alone misses defenders, goalkeeper positioning, shot type and match context. The 2015–16 season may not represent all leagues or seasons. The download requires an internet connection once; subsequent runs use the saved CSV. The data and scripts are intended for educational analysis; check the repository's license before redistribution.

**Public source:** https://github.com/statsbomb/open-data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("premier_league_2015_2016_shots.csv")

# The downloaded CSV is saved beside this notebook. Download it once if missing.
if not DATA.exists():
    print("CSV not found. Downloading the real StatsBomb shot events now...")
    import requests

    BASE = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"
    session = requests.Session()
    try:
        response = session.get(f"{BASE}/matches/2/27.json", timeout=25)
        response.raise_for_status()
        matches = response.json()
        records = []
        print(f"Found {len(matches)} available matches.", flush=True)
        for i, match in enumerate(matches, start=1):
            match_id = match["match_id"]
            response = session.get(f"{BASE}/events/{match_id}.json", timeout=35)
            response.raise_for_status()
            for event in response.json():
                if event.get("type", {}).get("name") != "Shot":
                    continue
                loc = event.get("location")
                outcome = event.get("shot", {}).get("outcome", {}).get("name")
                if not loc or len(loc) < 2 or not outcome:
                    continue
                records.append({
                    "event_id": event.get("id"),
                    "match_id": match_id,
                    "team": event.get("team", {}).get("name"),
                    "x": loc[0], "y": loc[1],
                    "goal": int(outcome == "Goal"),
                    "shot_outcome": outcome,
                    "shot_type": event.get("shot", {}).get("type", {}).get("name", "Unknown")
                })
            if i == 1 or i % 10 == 0 or i == len(matches):
                print(f"Processed {i}/{len(matches)} matches; {len(records)} shots", flush=True)
        if not records:
            raise ValueError("No shots found in the available matches.")
        pd.DataFrame(records).to_csv(DATA, index=False)
        print(f"Saved {DATA} ({len(records):,} shots)")
    except requests.RequestException as exc:
        raise RuntimeError(
            "Could not download StatsBomb data. Check your internet connection or "
            "place premier_league_2015_2016_shots.csv beside this notebook and rerun. "
            f"Original error: {exc}"
        ) from exc

shots = pd.read_csv(DATA)
required = {"x", "y", "goal", "match_id", "shot_outcome", "shot_type"}
missing = required - set(shots.columns)
if missing:
    raise ValueError(f"CSV is missing columns: {sorted(missing)}. Recreate it using the updated download script.")
print("Shots:", len(shots), "| Matches:", shots.match_id.nunique())
display(shots.head())


### Variables and cleaning

- `x`: shot start position along the pitch (StatsBomb 0–120 scale; larger x is closer to the attacking goal).
- `y`: shot start position across the pitch (StatsBomb 0–80 scale; 40 is the centerline).
- `goal`: binary categorical response, `1` for a goal, `0` for any other shot outcome.
- `match_id`: unique identifier of the match, used to check coverage.
- `team`: shooting team, retained for traceability, not used as a predictor.
- `shot_outcome`: original StatsBomb outcome category, retained for checking the binary recode.
- `shot_type`: penalty, free kick or open-play shot type, retained for transparency; no separate analysis is performed here.
- `distance_to_goal`: derived Euclidean distance to `(120, 40)` in StatsBomb coordinate units; useful for describing patterns, **not** a trained feature or model.

We exclude observations with invalid or missing shot coordinates or outcomes and report how many were removed. Penalties are retained in the overall analysis, but should be examined separately in future work because they occur from a fixed position.

In [ ]:
print("Missing values before cleaning:")
display(shots[["x", "y", "goal"]].isna().sum().rename("missing").to_frame())
original = len(shots)
for col in ["x", "y", "goal"]:
    shots[col] = pd.to_numeric(shots[col], errors="coerce")
shots = shots.dropna(subset=["x", "y", "goal"]).copy()
shots = shots.loc[shots.x.between(0, 120) & shots.y.between(0, 80) & shots.goal.isin([0, 1])].copy()
shots["goal"] = shots["goal"].astype(int)
shots["distance_to_goal"] = np.hypot(120 - shots.x, 40 - shots.y)
print("Rows excluded:", original - len(shots))
print("Duplicate event IDs:", shots.event_id.duplicated().sum() if "event_id" in shots else "Not available")
assert len(shots) > 0, "No valid shots in dataset"

## 3. Initial exploratory data analysis

First, how many shots became goals? How do the two coordinate variables vary? We use descriptive statistics and counts rather than fitting a predictive model.

In [ ]:
display(shots[["x", "y", "distance_to_goal", "goal"]].describe().round(2))
print("Goal outcome frequency table:")
display(shots.goal.map({0: "Not a goal", 1: "Goal"}).value_counts().rename_axis("outcome").to_frame("shots"))
print("Original shot outcome categories:")
display(shots.shot_outcome.value_counts(dropna=False).rename_axis("shot_outcome").to_frame("shots"))
print("Shot types:")
display(shots.shot_type.value_counts(dropna=False).rename_axis("shot_type").to_frame("shots"))
print(f"Overall observed goal rate: {shots.goal.mean():.1%}")

### Where are shots and goals taken?

The shot map uses the original StatsBomb coordinates, with the attacking goal on the right. A dense cluster indicates many attempts, but not necessarily a high scoring probability. Overlapping points are partly transparent.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for result, label, color, alpha in [(0, "Non-goal", "steelblue", .20), (1, "Goal", "crimson", .75)]:
    d = shots.loc[shots.goal == result]
    ax.scatter(d.x, d.y, s=14 if result == 0 else 25, alpha=alpha, c=color, label=f"{label} ({len(d):,})")
ax.plot([120, 120], [36, 44], color="black", lw=5, label="Goal mouth (schematic)")
ax.set(xlim=(0, 122), ylim=(80, 0), xlabel="Shot x (StatsBomb coordinates)", ylabel="Shot y (StatsBomb coordinates)", title="Premier League 2015–16: Shot locations")
ax.legend(loc="upper left"); ax.grid(alpha=.15); plt.show()

### Does observed scoring rate vary with distance?

Distance bands below use StatsBomb coordinate units, not meters The goal rate in a band is the number of goals divided by the number of shots in that band. Bands with few shots can have unstable rates; counts are displayed alongside percentages.

In [ ]:
bins = [0, 10, 20, 30, 40, 60, np.inf]
labels = ["0–10", "10–20", "20–30", "30–40", "40–60", "60+"]
shots["distance_band"] = pd.cut(shots.distance_to_goal, bins=bins, labels=labels, right=False)
summary = shots.groupby("distance_band", observed=False).agg(shots=("goal", "size"), goals=("goal", "sum"), goal_rate=("goal", "mean"))
display(summary.assign(goal_rate_pct=(100 * summary.goal_rate).round(1)))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(summary.index.astype(str), 100 * summary.goal_rate.fillna(0), color="steelblue")
ax.set(xlabel="Distance to attacking goal (StatsBomb coordinate units)", ylabel="Observed goals per 100 shots (%)", title="Observed goal rate by shot distance")
for i, row in enumerate(summary.itertuples()):
    ax.text(i, (row.goal_rate * 100 if pd.notna(row.goal_rate) else 0) + .4, f"n={row.shots}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

### Where do shots happen most often?

A two-dimensional histogram counts shots across locations. This plot shows volume, not scoring probability.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
h = ax.hist2d(shots.x, shots.y, bins=(24, 16), range=[[0, 120], [0, 80]], cmap="YlOrRd")
fig.colorbar(h[3], ax=ax, label="Number of shots")
ax.set(xlabel="Shot x", ylabel="Shot y", ylim=(80, 0), title="Shot volume by location")
plt.show()

### Initial findings and interpretation

The following summary is calculated from the real CSV when the notebook runs, so the text cannot accidentally report invented results. These patterns are descriptive associations, not proof that location causes goals.

In [ ]:
print(f"This dataset contains {len(shots):,} shots from {shots.match_id.nunique()} matches, including {shots.goal.sum():,} goals ({shots.goal.mean():.1%} of shots).")
valid = summary.loc[summary.shots > 0]
if len(valid) >= 2:
    nearest, farthest = valid.iloc[0], valid.iloc[-1]
    print(f"The nearest occupied distance band ({valid.index[0]}) has an observed goal rate of {nearest.goal_rate:.1%} from {nearest.shots} shots.")
    print(f"The farthest occupied distance band ({valid.index[-1]}) has an observed goal rate of {farthest.goal_rate:.1%} from {farthest.shots} shots.")
print("Compare the bar chart and shot map: higher shot volume does not automatically mean a higher conversion rate.")

## 4. Proposal for further study (no modeling performed)

My next step would be to build an expected goals (xG) model using logistic regression, because the response variable is binary and the model can output probabilities between 0 and 1. I would begin with the two shot coordinates, and possibly add distance and angle as derived location features. I would evaluate the predictions on matches held out from training, using calibration, log loss and Brier score. Splitting by match would help avoid leaking shots from the same game into both training and testing. In the future I also need to add more variables about the shot since there are more factors then just location

A larger question is whether location alone is enough to produce useful xG probabilities, or whether adding shot type, defensive pressure, body part and game situation would significantly improve accuracy. This could help fans and analysts compare shot quality, but a location-only model should not be used to make strong claims about individual player skill. The present notebook does not train, tune, or evaluate any model.

## Source and reproducibility

- StatsBomb Open Data: https://github.com/statsbomb/open-data
- Match index: https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/2/27.json
- Download/extraction script: `download_shots.py` (optional; the notebook can download the CSV on first run if internet access is available).
- CSV used in this notebook: `premier_league_2015_2016_shots.csv`.

**Attribution:** Data provided by StatsBomb. Check the source repository for terms of use and attribution requirements.